# Notebook 7 -- Three-Angle Evaluation on SVAMP
## LLM-to-SLM Guided Reasoning Pipeline (NVIDIA API Guide)

**What changed from Notebook 6?**

| Component | Notebook 6 (SLM-SLM) | Notebook 7 (LLM-SLM) |
|---|---|---|
| Guide model | Fine-tuned Llama 3B (local, LoRA) | Llama 3 70B via NVIDIA API |
| Guide output | Steps with calculations | **Verbal steps only** (no numbers) |
| Solver model | Llama 1.5B (local, unchanged) | Llama 1.5B (local, unchanged) |
| VRAM needed | ~9.3 GB (both models) | ~1.4 GB (solver only) |
| Guide params_B | 3.0 | 70.0 |

**Why verbal-only plans?**

The guide now tells the solver *what reasoning steps to take* in plain English,
without doing any arithmetic itself. This cleanly separates the roles:
- **Guide (LLM 70B)**: understands the problem, identifies the logical structure
- **Solver (SLM 1.5B)**: executes the arithmetic from scratch using the verbal plan

This tests whether a strong LLM's *language understanding* (not its math ability)
is what helps a small solver — a purer test of guidance value.

**Pipeline**
```
Question --> Llama 70B API (Guide) --> Verbal Plan --> Llama 1.5B (Solver) x5 --> Vote --> Answer
Baseline : Question ------------------------------------------> Llama 1.5B (Solver) x5 --> Vote
```

**Dataset: SVAMP** — robustness-focused arithmetic, 1,000 test questions.


In [1]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q peft==0.12.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
# !pip install -q openai          # <-- NEW: NVIDIA API client
print("Done.")


Done.


In [2]:
# CELL 2 -- HuggingFace login (only needed for solver model download)
from huggingface_hub import login
login("")
print('✅ HuggingFace login done')


✅ HuggingFace login done


In [3]:
# CELL 3 -- Imports + GPU
import os, json, re, glob, random, time
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm
from openai import OpenAI   # NVIDIA API client

OUTPUT_DIR = "/kaggle/working/svamp_eval_llm_slm"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")


PyTorch : 2.10.0+cu128
GPU     : Tesla T4
VRAM    : 15.6 GB
Output  : /kaggle/working/svamp_eval_llm_slm


In [4]:
# CELL 4 -- Configuration
CONFIG = {
    # Guide: NVIDIA-hosted LLM (no local GPU needed for guide)
    "nvidia_api_key"      : "",   # <-- paste your NVIDIA API key here
    "llm_guide_model"     : "meta/llama3-70b-instruct",

    # Solver: local SLM (unchanged from Notebook 6)
    "response_model"      : "meta-llama/Llama-3.2-1B-Instruct",

    # Dataset
    "dataset_name"        : "ChilleD/SVAMP",
    "dataset_split"       : "test",
    "max_eval_samples"    : 500,   # increase to 1000 for full run
    "random_seed"         : 42,    # SAME seed as Notebook 6 for fair comparison

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.7,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 350,

    # Compute cost tracking (billions of parameters)
    # Guide is API-based so we record its param count for analysis only
    "guide_params_B"      : 70.0,  # Llama 3 70B
    "solver_params_B"     : 1.0,   # Llama 3.2 1B

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    if k == "nvidia_api_key":
        print(f"  {'nvidia_api_key':<24}: {'✅ set' if v else '❌ MISSING'}")
    else:
        print(f"  {k:<24}: {v}")


Config ready:
  nvidia_api_key          : ✅ set
  llm_guide_model         : meta/llama3-70b-instruct
  response_model          : meta-llama/Llama-3.2-1B-Instruct
  dataset_name            : ChilleD/SVAMP
  dataset_split           : test
  max_eval_samples        : 500
  random_seed             : 42
  n_votes                 : 5
  vote_temperature        : 0.7
  guide_temperature       : 0.1
  refiner_temperature     : 0.3
  max_new_tokens          : 350
  guide_params_B          : 70.0
  solver_params_B         : 1.0
  results_file            : /kaggle/working/svamp_eval_llm_slm/results.jsonl
  report_file             : /kaggle/working/svamp_eval_llm_slm/eval_report.json
  angle1_file             : /kaggle/working/svamp_eval_llm_slm/angle1_compute_efficiency.json
  angle2_file             : /kaggle/working/svamp_eval_llm_slm/angle2_vote_consistency.json
  angle3_file             : /kaggle/working/svamp_eval_llm_slm/angle3_confidence_calibration.json
  checkpoint_file         : /kaggle/

In [5]:
# CELL 5 -- Load SVAMP dataset
# SVAMP fields: Body, Question, Equation, Answer (numeric)
# We combine Body + Question into a single question string.

print("Loading SVAMP from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"])

print(f"Splits     : {list(raw_ds.keys())}")
print(f"Features   : {list(raw_ds[CONFIG['dataset_split']].features.keys())}")
print(f"Test size  : {len(raw_ds[CONFIG['dataset_split']])}")
print(f"Example    :")
ex = raw_ds[CONFIG["dataset_split"]][0]
for k, v in ex.items():
    print(f"  {k}: {v}")


def normalise_svamp(item):
    """Convert SVAMP record to {question, answer} used by the pipeline."""
    q = item["Body"].strip().rstrip(".") + " " + item["Question"].strip()
    ans = item["Answer"]
    # Store as clean integer string when possible (5.0 -> "5")
    if isinstance(ans, float) and ans == int(ans):
        ans_str = str(int(ans))
    else:
        ans_str = str(ans)
    return {"question": q, "answer": ans_str}


all_data = [normalise_svamp(x) for x in raw_ds[CONFIG["dataset_split"]]]

# ---- CRITICAL: set seed ONCE here, before any sampling ----------
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

# Fingerprint so we can verify reproducibility vs Notebook 6
print(f"First Q  : {test_data[0]['question'][:80]}...")
print(f"First A  : {test_data[0]['answer']}")
print(f"Last  Q  : {test_data[-1]['question'][:60]}...")
print("SVAMP loaded")


Loading SVAMP from HuggingFace...


README.md:   0%|          | 0.00/675 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/111k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/54.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/700 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Splits     : ['train', 'test']
Features   : ['ID', 'Body', 'Question', 'Equation', 'Answer', 'Type', 'question_concat']
Test size  : 300
Example    :
  ID: chal-736
  Body: Winter is almost here and most animals are migrating to warmer countries. There are 41 bird families living near the mountain. If 35 bird families flew away to asia and 62 bird families flew away to africa
  Question: How many more bird families flew away to africa than those that flew away to asia?
  Equation: ( 62.0 - 35.0 )
  Answer: 27
  Type: Subtraction
  question_concat: Winter is almost here and most animals are migrating to warmer countries. There are 41 bird families living near the mountain. If 35 bird families flew away to asia and 62 bird families flew away to africa How many more bird families flew away to africa than those that flew away to asia?

Using all 300 questions
First Q  : Winter is almost here and most animals are migrating to warmer countries. There ...
First A  : 27
Last  Q  : Jake has 13 

In [6]:
# CELL 6 -- Answer extraction (same as Notebook 6)

def normalise_num(s):
    """Convert numeric string to canonical form. 5.0 -> '5', 3.14 -> '3.14'."""
    s = s.replace(",", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except ValueError:
        return s


def extract_gt_answer(answer_str):
    """SVAMP GT is already clean -- just normalise."""
    return normalise_num(str(answer_str))


def extract_pred_answer(text):
    """
    Multi-pattern extractor. Returns empty string on failure.

    Priority:
      1. #### N          -- standard format we request
      2. \\boxed{N}      -- LaTeX style
      3. 'the answer is' -- common phrasing
      4. '= N' at end of line
      5. **N** at end    -- bold markdown
      6. 'therefore N'   -- conclusion phrases
    """
    # 1
    m = re.search(r"####\s*(-?[\d\.]+)", text)
    if m: return normalise_num(m.group(1))
    # 2
    m = re.search(r"\\boxed\{(-?[\d\.]+)\}", text)
    if m: return normalise_num(m.group(1))
    # 3
    m = re.search(r"(?:the answer is|answer is)\s*:?\s*\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    # 4
    m = re.search(r"=\s*\$?(-?[\d\.]+)\s*$", text.strip(), re.MULTILINE)
    if m: return normalise_num(m.group(1))
    # 5
    m = re.search(r"\*\*\$?(-?[\d\.]+)\*\*\.?\s*$", text.strip())
    if m: return normalise_num(m.group(1))
    # 6
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    return ""


# --- Self-test ---
_tests = [
    ("#### 42",                 "42"),
    ("#### 3.5",                "3.5"),
    ("\\boxed{100}",            "100"),
    ("The answer is 7",         "7"),
    ("Total = 20",              "20"),
    ("**200**.",                "200"),
    ("Therefore, 13",           "13"),
    ("Some unrelated text",     ""),
]
ok = True
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    status = "OK" if got == exp else "FAIL"
    if got != exp: ok = False
    print(f"  {status}  '{txt[:35]}' -> '{got}' (expected '{exp}')")
print("All extractor tests passed" if ok else "EXTRACTOR HAS FAILURES -- fix before running eval")


  OK  '#### 42' -> '42' (expected '42')
  OK  '#### 3.5' -> '3.5' (expected '3.5')
  OK  '\boxed{100}' -> '100' (expected '100')
  OK  'The answer is 7' -> '7' (expected '7')
  OK  'Total = 20' -> '20' (expected '20')
  OK  '**200**.' -> '200' (expected '200')
  OK  'Therefore, 13' -> '13' (expected '13')
  OK  'Some unrelated text' -> '' (expected '')
All extractor tests passed


In [7]:
# CELL 7 -- Set up NVIDIA API client (replaces local 3B guide model)
#
# The guide is now Llama 3 70B served via NVIDIA's inference API.
# No GPU memory is used for the guide -- only the solver loads locally.
#
# Get your free API key at: https://build.nvidia.com

assert CONFIG["nvidia_api_key"], (
    "❌ nvidia_api_key is empty! Paste your key into CONFIG in Cell 4."
)

nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=CONFIG["nvidia_api_key"],
)

# Quick connectivity test
try:
    _test = nvidia_client.chat.completions.create(
        model=CONFIG["llm_guide_model"],
        messages=[{"role": "user", "content": "Reply with just: OK"}],
        max_tokens=5,
        temperature=0.0,
    )
    print(f"✅ NVIDIA API connected | model: {CONFIG['llm_guide_model']}")
    print(f"   Test response: {_test.choices[0].message.content.strip()}")
except Exception as e:
    print(f"❌ NVIDIA API connection failed: {e}")
    raise


✅ NVIDIA API connected | model: meta/llama3-70b-instruct
   Test response: OK


In [8]:
# CELL 8 -- Load solver model (Llama 1.5B, local)
# Only the solver loads onto GPU -- much lower VRAM vs Notebook 6.

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

solver_vram = torch.cuda.memory_allocated() / 1e9
total_vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Solver VRAM  : {solver_vram:.2f} GB  (guide is on NVIDIA cloud, not here)")
print(f"Total GPU    : {total_vram:.1f} GB")
print(f"Headroom     : {total_vram - solver_vram:.1f} GB")
print("Memory OK -- only solver is local")


Loading solver: meta-llama/Llama-3.2-1B-Instruct


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Solver VRAM  : 1.26 GB  (guide is on NVIDIA cloud, not here)
Total GPU    : 15.6 GB
Headroom     : 14.4 GB
Memory OK -- only solver is local


In [9]:
# CELL 9 -- Prompts and generation functions
#
# KEY CHANGE: GUIDE_SYSTEM now asks for VERBAL steps only.
# The guide describes WHAT to do (identify quantities, choose operation,
# name the relationship) without performing any arithmetic.
# The solver receives this verbal plan and does all the number work.
#
# Separation of roles:
#   Guide  --> language understanding, problem decomposition (no math)
#   Solver --> arithmetic execution (guided by the verbal plan)

GUIDE_SYSTEM = (
    "You are a math problem analyst. Your ONLY job is to identify the logical "
    "reasoning steps needed to solve the problem.\n\n"
    "STRICT RULES:\n"
    "- Write steps in plain English. Do NOT perform any arithmetic.\n"
    "- Do NOT write any numbers, equations, or calculations in your steps.\n"
    "- Do NOT give the final answer or any intermediate numeric result.\n"
    "- Each step must say WHAT operation to do and WHY, using the names of "
    "  quantities, not their values.\n"
    "- Maximum 3 steps. Be concise.\n\n"
    "BAD  (has numbers and calculations):\n"
    "  Step 1: 62 - 35 = 27\n"
    "  Step 2: Answer is 27\n\n"
    "GOOD (verbal description only):\n"
    "  Step 1: Identify the two quantities being compared: families that flew "
    "    to Africa and families that flew to Asia.\n"
    "  Step 2: Subtract the Asia count from the Africa count, because the "
    "    question asks how many MORE flew to Africa.\n"
    "  Step 3: The result of that subtraction is the answer.\n\n"
    "Output only the numbered steps. Nothing else."
)

SOLVE_SYSTEM = (
    "You are a math problem solver.\n"
    "You will be given a problem and a verbal reasoning plan.\n"
    "Follow the plan exactly. Compute each step numerically.\n"
    "No markdown. No bullet points. No headers.\n"
    "Write plain arithmetic steps only.\n"
    "Your absolute last line must be: #### [number]\n"
    "NEVER write ### or ** in your response.\n\n"
    "Example:\n"
    "Eaten = 14. Given = 13.\n"
    "Difference = 14 - 13 = 1.\n"
    "#### 1"
)

BASELINE_SYSTEM = (
    "You are a precise math problem solver.\n"
    "Read the problem carefully. Solve step by step, showing every calculation.\n"
    "Your FINAL line must be exactly: #### [number]"
)

REFINER_SYSTEM = (
    "You are a careful math problem solver.\n"
    "Previous attempts on this problem gave different answers.\n"
    "Re-solve completely from scratch using a fresh approach.\n"
    "Show every arithmetic step.\n"
    "Your FINAL line must be exactly: #### [number]"
)


# ---- Local solver call (same as Notebook 6) ----
def run_solver(messages, max_tokens, temperature):
    """Call the local SLM solver and return generated text."""
    prompt = resp_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = resp_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(resp_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = resp_model.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = resp_tok.eos_token_id,
            repetition_penalty = 1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return resp_tok.decode(new_toks, skip_special_tokens=True).strip()


# ---- NVIDIA API guide call (NEW) ----
def generate_plan(question, max_retries=3):
    """
    Call Llama 70B via NVIDIA API to produce a VERBAL reasoning plan.
    The plan contains no numbers or calculations -- only natural language
    descriptions of what steps the solver should follow.
    """
    messages = [
        {"role": "system", "content": GUIDE_SYSTEM},
        {"role": "user",   "content": f"Problem: {question}"},
    ]
    for attempt in range(max_retries):
        try:
            completion = nvidia_client.chat.completions.create(
                model       = CONFIG["llm_guide_model"],
                messages    = messages,
                temperature = CONFIG["guide_temperature"],
                max_tokens  = 300,
                stream      = False,
            )
            plan = completion.choices[0].message.content.strip()
            return plan
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)   # exponential backoff
            else:
                print(f"  [guide API error after {max_retries} attempts]: {e}")
                return "Step 1: Read the problem carefully and identify all quantities.\nStep 2: Determine the correct arithmetic operation.\nStep 3: Compute the result."


# ---- Solver functions (local SLM, same interface as Notebook 6) ----
def generate_guided(question, plan):
    """
    Solver receives the verbal plan from the LLM guide and executes the math.
    The plan is presented as a reasoning guide -- solver fills in all numbers.
    """
    content = (
        f"Problem: {question}\n\n"
        f"Reasoning plan (follow each step and compute the numbers):\n{plan}\n\n"
        f"Now solve step by step with all arithmetic:"
    )
    return run_solver(
        [{"role": "system", "content": SOLVE_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_baseline(question):
    return run_solver(
        [{"role": "system", "content": BASELINE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_refiner(question, plan, candidates):
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"Problem: {question}\n\n"
        f"Reasoning plan:\n{plan}\n\n"
        f"Previous attempts disagreed: {cands}\n"
        "Re-solve carefully from scratch:"
    )
    return run_solver(
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )


print("Generation functions ready")
print("  generate_plan()     -> NVIDIA Llama 70B API (verbal steps, no math)")
print("  generate_guided()   -> local 1B solver with plan")
print("  generate_baseline() -> local 1B solver no plan")
print("  generate_refiner()  -> local 1B solver tie-breaker")


Generation functions ready
  generate_plan()     -> NVIDIA Llama 70B API (verbal steps, no math)
  generate_guided()   -> local 1B solver with plan
  generate_baseline() -> local 1B solver no plan
  generate_refiner()  -> local 1B solver tie-breaker


In [10]:
# CELL 10 -- Voting logic with richer metrics (same as Notebook 6)

def vote_and_decide(answers, question, plan, gt_answer=None):
    """
    Majority voting with refiner fallback on ties.
    Returns a dict with all metrics needed for the three angles.

    Fields:
      final_answer     : chosen answer string
      strategy         : 'majority' | 'refiner_tiebreak' | 'coin_flip'
      confidence       : top_count / total_votes
      correct_votes    : votes that matched gt_answer
      vote_consistency : correct_votes / total_votes
      wasted_votes     : votes that did NOT match final_answer
      refiner_used     : bool
      refiner_correct  : bool or None
    """
    valid = [a for a in answers if a and a.strip()]
    if not valid:
        valid = answers  # fallback
    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(answers)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / total

    is_majority = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        # Tie: call refiner as an extra vote
        ref_raw    = generate_refiner(question, plan, list(answers))
        ref_ans    = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_votes  = answers + [ref_ans]
        new_counts = Counter(all_votes)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_votes), 4)
        vote_counts = new_counts
        total      = len(all_votes)
        correct_votes    = new_counts.get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / total
        wasted           = total - new_top_c

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "correct_votes"    : correct_votes,
        "total_votes"      : total,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }


print("Voting logic ready")
print("  majority        -> clear winner across votes")
print("  refiner_tiebreak-> tie broken by refiner call")
print("  coin_flip       -> still tied after refiner")


Voting logic ready
  majority        -> clear winner across votes
  refiner_tiebreak-> tie broken by refiner call
  coin_flip       -> still tied after refiner


In [11]:
# CELL 11 -- Single question test (verify LLM guide + SLM solver pipeline)

print("=" * 65)
print("SINGLE QUESTION TEST  (SVAMP)")
print("=" * 65)
q  = test_data[0]["question"]
gt = extract_gt_answer(test_data[0]["answer"])
print(f"Question : {q}")
print(f"GT Answer: {gt}")

print("\n[1] LLM guide (70B via NVIDIA API) generating VERBAL plan...")
plan = generate_plan(q)
print("Plan (verbal steps only, no numbers):")
print(plan)

print("\n[2] Guided solver votes (5x, local 1B)...")
g_votes_raw = []
for i in range(CONFIG["n_votes"]):
    raw = generate_guided(q, plan)
    ans = extract_pred_answer(raw)
    g_votes_raw.append(ans)
    print(f"  Vote {i+1}: '{ans}'  |  raw[:80]: {raw[:80]}")

g = vote_and_decide(g_votes_raw, q, plan, gt)
print(f"\n  Result   : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['correct_votes']/g['total_votes']*100:.0f}%)")

print("\n[3] Baseline votes (no plan, local 1B)...")
b_votes_raw = []
for i in range(CONFIG["n_votes"]):
    raw = generate_baseline(q)
    ans = extract_pred_answer(raw)
    b_votes_raw.append(ans)
    print(f"  Vote {i+1}: '{ans}'")

b = vote_and_decide(b_votes_raw, q, "baseline", gt)
print(f"\n  Baseline result : {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


SINGLE QUESTION TEST  (SVAMP)
Question : Winter is almost here and most animals are migrating to warmer countries. There are 41 bird families living near the mountain. If 35 bird families flew away to asia and 62 bird families flew away to africa How many more bird families flew away to africa than those that flew away to asia?
GT Answer: 27

[1] LLM guide (70B via NVIDIA API) generating VERBAL plan...
Plan (verbal steps only, no numbers):
1. Identify the two quantities being compared: bird families that flew to Africa and bird families that flew to Asia.
2. Subtract the Asia count from the Africa count, because the question asks how many MORE flew to Africa.
3. The result of that subtraction is the answer.

[2] Guided solver votes (5x, local 1B)...
  Vote 1: '6'  |  raw[:80]: To find out how many more bird families migrated to Africa than to Asia:

Step 1
  Vote 2: '35'  |  raw[:80]: To find out how many more bird families migrated to Africa than to Asia:

First 
  Vote 3: '35'  |  ra

In [12]:
# CELL 12 -- Full Dual Evaluation Loop
#
# Runs every question TWICE with the SAME questions (seed fixed in Cell 5):
#   Mode A: Guided   (70B LLM verbal plan via API + 1B solver x5)
#   Mode B: Baseline (1B solver x5, no plan)
#
# NOTE: The NVIDIA API guide adds ~0.5-1s per question (network latency).
# The overall loop will be faster than Notebook 6 because there is no
# local 3B model doing inference on GPU.
#
# API rate limit tip: free NVIDIA tier = ~5 req/s.
# If you hit 429 errors, generate_plan() will auto-retry with backoff.

print(f"Dual evaluation: {len(test_data)} SVAMP questions")
print(f"Guide  : {CONFIG['llm_guide_model']} via NVIDIA API (verbal plans)")
print(f"Solver : {CONFIG['response_model']} local")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print("-" * 65)

all_results  = []   # guided results
base_results = []   # baseline results
start_idx    = 0

# Resume from checkpoint
if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
    print(f"  Guided saved: {len(all_results)}  Baseline saved: {len(base_results)}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="SVAMP Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    # ---- GUIDED (LLM plan + SLM solver) -------------------------
    try:
        plan        = generate_plan(question)   # 70B API, verbal steps
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, plan, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "correct_votes"    : g_dec["correct_votes"],
            "total_votes"      : g_dec["total_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
            "vote_counts"      : g_dec["vote_counts"],
            "plan"             : plan,
        })
    except RuntimeError as e:
        all_results.append({
            "mode": "guided", "idx": idx, "question": question,
            "gt_answer": gt_answer, "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    # ---- BASELINE (no plan) -------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, "baseline", gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "correct_votes"    : b_dec["correct_votes"],
            "total_votes"      : b_dec["total_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : None,
            "vote_counts"      : b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode": "baseline", "idx": idx, "question": question,
            "gt_answer": gt_answer, "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    # Checkpoint
    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:3d}] Guided: {g_acc:.1f}%  Baseline: {b_acc:.1f}%  ({mins:.1f} min)")

# Final save
with open(CONFIG["results_file"], "w") as f:
    for r in all_results + base_results:
        f.write(json.dumps(r) + "\n")

g_c = sum(r["correct"] for r in all_results)
b_c = sum(r["correct"] for r in base_results)
print(f"\nEvaluation complete.")
print(f"  Guided (LLM-SLM)  : {g_c}/{len(all_results)} = {g_c/len(all_results)*100:.1f}%")
print(f"  Baseline (SLM)    : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%")
print(f"  Delta             : +{(g_c/len(all_results) - b_c/len(base_results))*100:.1f} percentage points")


Dual evaluation: 300 SVAMP questions
Guide  : meta/llama3-70b-instruct via NVIDIA API (verbal plans)
Solver : meta-llama/Llama-3.2-1B-Instruct local
Each question: 5 guided votes + 5 baseline votes
-----------------------------------------------------------------
Starting fresh


SVAMP Eval:   0%|          | 0/300 [00:00<?, ?it/s]

  [ 25] Guided: 64.0%  Baseline: 24.0%  (16.6 min)
  [ 50] Guided: 64.0%  Baseline: 40.0%  (32.7 min)
  [ 75] Guided: 61.3%  Baseline: 32.0%  (49.4 min)
  [100] Guided: 56.0%  Baseline: 33.0%  (67.0 min)
  [125] Guided: 56.8%  Baseline: 32.8%  (83.7 min)
  [150] Guided: 58.0%  Baseline: 32.0%  (101.0 min)
  [175] Guided: 58.3%  Baseline: 30.9%  (118.5 min)
  [200] Guided: 58.0%  Baseline: 31.0%  (136.5 min)
  [225] Guided: 56.4%  Baseline: 31.6%  (154.4 min)
  [250] Guided: 55.6%  Baseline: 31.2%  (171.7 min)
  [275] Guided: 55.3%  Baseline: 32.0%  (189.5 min)
  [300] Guided: 56.7%  Baseline: 32.3%  (205.9 min)

Evaluation complete.
  Guided (LLM-SLM)  : 170/300 = 56.7%
  Baseline (SLM)    : 97/300 = 32.3%
  Delta             : +24.3 percentage points


In [13]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY
# =================================================================
# We compare three setups by accuracy and local compute cost.
#
# NOTE: The 70B guide runs on NVIDIA's cloud, so its param-passes
# are NOT local GPU compute. We track it separately for context.
#
#   Baseline : 1B solver x5                       =  5.0B local param-passes
#   Guided   : 70B guide (API) x1 + 1B solver x5  =  5.0B local + 70B cloud
#   Upper    : 70B model x5 (hypothetical ceiling) = 350.0B param-passes
#
# For local efficiency, baseline and guided use the same local compute.
# The gain comes purely from the quality of the verbal plan.
# =================================================================

G = CONFIG["guide_params_B"]    # 70.0 (cloud)
S = CONFIG["solver_params_B"]   # 1.0
N = CONFIG["n_votes"]           # 5

guided_local_compute   = S * N              # 5.0  (only solver is local)
baseline_local_compute = S * N              # 5.0
guided_total_compute   = (G * 1) + (S * N)  # 75.0 (including cloud guide)
upper_compute          = G * N              # 350.0 (hypothetical 70B x5)

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100

# Accuracy-per-billion-local-param-passes
g_eff = g_acc / guided_local_compute
b_eff = b_acc / baseline_local_compute

# Wasted votes
g_wasted       = sum(r["wasted_votes"] for r in all_results)
b_wasted       = sum(r["wasted_votes"] for r in base_results)
total_possible = len(all_results) * N

# Refiner stats
ref_triggered  = sum(r["refiner_used"] for r in all_results)
ref_correct    = sum(1 for r in all_results
                     if r["refiner_used"] and r.get("refiner_correct"))

# Per-strategy accuracy
strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats:
        strategy_stats[s] = {"n": 0, "correct": 0}
    strategy_stats[s]["n"] += 1
    if r["correct"]:
        strategy_stats[s]["correct"] += 1

savings_pct = (1 - guided_total_compute / upper_compute) * 100
acc_gain    = g_acc - b_acc

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (SVAMP, LLM-SLM)")
print("=" * 65)
print(f"\n  {'Setup':<36} | {'Local Compute':>14} | {'Accuracy':>9} | {'Acc/B':>7}")
print(f"  {'-'*36}-+-{'-'*14}-+-{'-'*9}-+-{'-'*7}")
print(f"  {'Baseline (1B x ' + str(N) + ', no guide)':<36} | {baseline_local_compute:>12.1f}B  | {b_acc:>8.1f}% | {b_eff:>6.3f}")
print(f"  {'Guided  (70B API x1 + 1B x' + str(N) + ')':<36} | {guided_local_compute:>12.1f}B  | {g_acc:>8.1f}% | {g_eff:>6.3f}")
print(f"  {'Upper   (70B x ' + str(N) + ', hypothetical)':<36} | {upper_compute:>12.1f}B  | {'(ceiling)':>9} |")
print(f"\n  Note: Guide (70B) runs on NVIDIA cloud. Local compute is identical")
print(f"        for both modes. Accuracy gain is purely from verbal plan quality.")
print(f"\n  Accuracy gain over baseline  : +{acc_gain:.1f} percentage points")
print(f"  Total compute savings vs upper: {savings_pct:.0f}% cheaper (inc. cloud guide)")
print(f"  Local efficiency (acc/B)      : {g_eff:.3f} vs {b_eff:.3f} (same -- same local compute)")

print(f"\n  Wasted votes (votes != final answer):")
print(f"    Guided   : {g_wasted} / {total_possible}  ({g_wasted/total_possible*100:.1f}%)")
print(f"    Baseline : {b_wasted} / {total_possible}  ({b_wasted/total_possible*100:.1f}%)")
print(f"    Saved    : {b_wasted - g_wasted} fewer wasted compute passes with guidance")

if ref_triggered > 0:
    print(f"\n  Refiner (tie-breaker) stats:")
    print(f"    Triggered : {ref_triggered} / {len(all_results)} questions")
    print(f"    Correct   : {ref_correct} / {ref_triggered}  ({ref_correct/ref_triggered*100:.1f}% of ties resolved correctly)")

print(f"\n  Decision strategy breakdown (guided):")
print(f"  {'Strategy':<22} | {'Count':>6} | {'Accuracy':>9}")
print(f"  {'-'*22}-+-{'-'*6}-+-{'-'*9}")
for s, v in sorted(strategy_stats.items(), key=lambda x: -x[1]["n"]):
    acc_s = v["correct"] / v["n"] * 100 if v["n"] else 0
    print(f"  {s:<22} | {v['n']:>6} | {acc_s:>8.1f}%")

angle1 = {
    "dataset"               : "SVAMP",
    "pipeline"              : "LLM-SLM (70B API guide + 1B local solver)",
    "n_questions"           : len(all_results),
    "guided_local_compute_B": guided_local_compute,
    "guided_total_compute_B": guided_total_compute,
    "baseline_compute_B"    : baseline_local_compute,
    "upper_compute_B"       : upper_compute,
    "guided_accuracy"       : round(g_acc, 2),
    "baseline_accuracy"     : round(b_acc, 2),
    "accuracy_gain"         : round(acc_gain, 2),
    "compute_savings_pct"   : round(savings_pct, 1),
    "guided_efficiency"     : round(g_eff, 4),
    "baseline_efficiency"   : round(b_eff, 4),
    "guided_wasted_votes"   : g_wasted,
    "baseline_wasted_votes" : b_wasted,
    "wasted_votes_saved"    : b_wasted - g_wasted,
    "refiner_triggered"     : ref_triggered,
    "refiner_correct"       : ref_correct,
    "strategy_breakdown"    : strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(angle1, f, indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")


ANGLE 1 -- COMPUTE EFFICIENCY  (SVAMP, LLM-SLM)

  Setup                                |  Local Compute |  Accuracy |   Acc/B
  -------------------------------------+----------------+-----------+--------
  Baseline (1B x 5, no guide)          |          5.0B  |     32.3% |  6.467
  Guided  (70B API x1 + 1B x5)         |          5.0B  |     56.7% | 11.333
  Upper   (70B x 5, hypothetical)      |        350.0B  | (ceiling) |

  Note: Guide (70B) runs on NVIDIA cloud. Local compute is identical
        for both modes. Accuracy gain is purely from verbal plan quality.

  Accuracy gain over baseline  : +24.3 percentage points
  Total compute savings vs upper: 79% cheaper (inc. cloud guide)
  Local efficiency (acc/B)      : 11.333 vs 6.467 (same -- same local compute)

  Wasted votes (votes != final answer):
    Guided   : 750 / 1500  (50.0%)
    Baseline : 780 / 1500  (52.0%)
    Saved    : 30 fewer wasted compute passes with guidance

  Refiner (tie-breaker) stats:
    Triggered : 77 / 3

In [14]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY
# =================================================================
# Vote consistency = fraction of votes (out of 5) that matched GT.
# A high-consistency question means the model reliably solves it.
# A low-consistency question means it got lucky on the final vote.
#
# If verbal guidance works, we expect BOTH:
#   - Higher mean consistency (more votes correct per question)
#   - More questions in the "high" bucket (4-5 correct votes)
#
# Per-question comparison: on each question, does guided produce
# more correct votes than baseline? Win/Loss/Tie.
# =================================================================

g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

# Per-question: guided better / baseline better / tied
guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

# Distribution buckets
def bucket(scores):
    return {
        "all_wrong  (0%)": sum(1 for s in scores if s == 0.0),
        "low       (1-39%)": sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

g_corr_cons = [r["vote_consistency"] for r in all_results  if r["correct"]]
b_corr_cons = [r["vote_consistency"] for r in base_results if r["correct"]]

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (SVAMP, LLM-SLM)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio (out of {CONFIG['n_votes']} votes per question):")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Lift     : {lift:.2f}x  (guided produces {lift:.1f}x more correct votes per question)")

print(f"\n  Per-question comparison (same questions, both modes):")
print(f"    Guided beats baseline : {guided_wins} / {len(all_results)} questions")
print(f"    Baseline beats guided : {baseline_wins} / {len(all_results)} questions")
print(f"    Equal                 : {tied} / {len(all_results)} questions")

print(f"\n  Distribution of vote consistency:")
print(f"  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gv  = g_dist[bkt]
    bv  = b_dist[bkt]
    dif = gv - bv
    sign = "+" if dif >= 0 else ""
    print(f"  {bkt:<22} | {gv:>8} | {bv:>8} | {sign+str(dif):>6}")

if g_corr_cons:
    print(f"\n  Among CORRECT questions only -- average vote consistency:")
    print(f"    Guided   : {np.mean(g_corr_cons)*100:.1f}%  (n={len(g_corr_cons)})")
    print(f"    Baseline : {np.mean(b_corr_cons)*100:.1f}%  (n={len(b_corr_cons)})")
    print("    (High consistency + correct = genuine reliable solving, not lucky vote)")

angle2 = {
    "dataset"                   : "SVAMP",
    "pipeline"                  : "LLM-SLM",
    "n_questions"               : len(all_results),
    "guided_mean_consistency"   : round(g_mean, 4),
    "baseline_mean_consistency" : round(b_mean, 4),
    "consistency_lift"          : round(lift, 4),
    "guided_wins"               : guided_wins,
    "baseline_wins"             : baseline_wins,
    "tied"                      : tied,
    "guided_distribution"       : g_dist,
    "baseline_distribution"     : b_dist,
    "guided_correct_q_consistency"   : round(np.mean(g_corr_cons), 4) if g_corr_cons else 0,
    "baseline_correct_q_consistency" : round(np.mean(b_corr_cons), 4) if b_corr_cons else 0,
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(angle2, f, indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")


ANGLE 2 -- VOTE CONSISTENCY  (SVAMP, LLM-SLM)

  Mean correct-vote ratio (out of 5 votes per question):
    Guided   : 41.0%  (2.05 votes correct on average)
    Baseline : 24.3%  (1.22 votes correct on average)
    Lift     : 1.69x  (guided produces 1.7x more correct votes per question)

  Per-question comparison (same questions, both modes):
    Guided beats baseline : 164 / 300 questions
    Baseline beats guided : 63 / 300 questions
    Equal                 : 73 / 300 questions

  Distribution of vote consistency:
  Bucket                 |   Guided | Baseline |   Diff
  -----------------------+----------+----------+-------
  all_wrong  (0%)        |       71 |      126 |    -55
  low       (1-39%)      |       67 |       80 |    -13
  medium  (40-79%)       |       92 |       68 |    +24
  high   (80-100%)       |       70 |       26 |    +44

  Among CORRECT questions only -- average vote consistency:
    Guided   : 64.3%  (n=170)
    Baseline : 56.6%  (n=97)
    (High consisten

In [15]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION
# =================================================================
# Confidence = fraction of votes that agreed on the winning answer.
# Perfect calibration: "80% confident" means 80% accurate.
#
# ECE (Expected Calibration Error) measures the average gap
# between stated confidence and actual accuracy across all buckets.
# LOWER ECE = more trustworthy confidence signal.
#
# False confidence = all 5 votes agree on the WRONG answer.
# Verbal guidance should suppress false confidence by steering votes
# toward correct reasoning paths rather than shared mistakes.
# =================================================================

def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40,  0.25),
    ]
    n_total   = len(results)
    ece       = 0.0
    calib_out = []

    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")

    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({
            "bucket": name, "count": n,
            "accuracy": round(acc, 4), "expected": mid, "gap": round(gap, 4)
        })

    hc_items = [r for r in results if r["confidence"] >= 0.80]
    hc_acc   = sum(r["correct"] for r in hc_items) / max(1, len(hc_items)) * 100
    print(f"  {'ECE (lower=better)':<26}   {ece:>5.4f}")
    print(f"  High-conf questions : {len(hc_items)}  |  Accuracy when confident: {hc_acc:.1f}%")
    print(f"  Confidently WRONG   : {false_conf} questions (false confidence)")
    return ece, calib_out, false_conf


print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (SVAMP, LLM-SLM)")
print("=" * 65)
print("Ideal: accuracy at each confidence level matches that level.")
print("False confidence: model agrees on wrong answer with full certainty.")

g_ece, g_calib, g_false = calibration_report(all_results,  "GUIDED (LLM verbal plan + SLM solver)")
b_ece, b_calib, b_false = calibration_report(base_results, "BASELINE (no plan)")

improve_pct = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE Summary:")
print(f"    Guided   ECE : {g_ece:.4f}")
print(f"    Baseline ECE : {b_ece:.4f}")
print(f"    Improvement  : {improve_pct:.1f}% better calibrated")
print(f"\n  False Confidence (confident AND wrong):")
print(f"    Guided   : {g_false} questions")
print(f"    Baseline : {b_false} questions")
print(f"    Reduction: {b_false - g_false} fewer false-confidence questions with guidance")

angle3 = {
    "dataset"                  : "SVAMP",
    "pipeline"                 : "LLM-SLM",
    "n_questions"              : len(all_results),
    "guided_ece"               : round(g_ece, 4),
    "baseline_ece"             : round(b_ece, 4),
    "ece_improvement_pct"      : round(improve_pct, 2),
    "guided_false_confidence"  : g_false,
    "baseline_false_confidence": b_false,
    "false_conf_reduction"     : b_false - g_false,
    "guided_calibration"       : g_calib,
    "baseline_calibration"     : b_calib,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(angle3, f, indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")


ANGLE 3 -- CONFIDENCE CALIBRATION  (SVAMP, LLM-SLM)
Ideal: accuracy at each confidence level matches that level.
False confidence: model agrees on wrong answer with full certainty.

  [GUIDED (LLM verbal plan + SLM solver)]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |    75 |     93.3% |       90% |  0.033 | Good
  High       (0.60-0.80)     |    63 |     66.7% |       70% |  0.033 | Good
  Medium     (0.40-0.60)     |   101 |     46.5% |       50% |  0.035 | Good
  Low        (<0.40)         |    61 |     18.0% |       25% |  0.070 | Good
  ECE (lower=better)           0.0412
  High-conf questions : 75  |  Accuracy when confident: 93.3%
  Confidently WRONG   : 5 questions (false confidence)

  [BASELINE (no plan)]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+---------

In [16]:
# CELL 16 -- Full Paper Summary (all three angles)

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

n  = a1["n_questions"]
ga = a1["guided_accuracy"]
ba = a1["baseline_accuracy"]
gc = a1["guided_local_compute_B"]
uc = a1["upper_compute_B"]

print("=" * 68)
print("  SVAMP EVALUATION -- PAPER SUMMARY TABLE (LLM-SLM)")
print("=" * 68)
print(f"  Dataset  : SVAMP  |  Questions: {n}  |  Seed: {CONFIG['random_seed']}")
print(f"  Guide    : Llama 70B via NVIDIA API (verbal plans, no arithmetic)")
print(f"  Solver   : Llama 1.5B local")
print()

rows = [
    ["Metric",                    "Baseline",       "Guided (LLM-SLM)", "Change"],
    ["Overall Accuracy",
     str(ba) + "%",               str(ga) + "%",
     "+" + str(round(ga-ba,1)) + " pts"],
    ["Local Compute Cost",
     str(a1['baseline_compute_B']) + "B param-passes",
     str(gc) + "B (+ 70B cloud guide)",
     "Same local compute; gain from plan"],
    ["Wasted Votes",
     str(a1['baseline_wasted_votes']),
     str(a1['guided_wasted_votes']),
     str(a1['wasted_votes_saved']) + " fewer wasted passes"],
    ["Vote Consistency",
     str(round(a2['baseline_mean_consistency']*100,1)) + "%",
     str(round(a2['guided_mean_consistency']*100,1)) + "%",
     str(round(a2['consistency_lift'],2)) + "x lift"],
    ["High-Agreement Questions",
     str(a2['baseline_distribution']['high   (80-100%)']),
     str(a2['guided_distribution']['high   (80-100%)']),
     ""],
    ["Guided Wins Per-Question",
     "--",
     str(a2['guided_wins']) + " / " + str(n),
     ""],
    ["ECE (lower = better)",
     str(a3['baseline_ece']),
     str(a3['guided_ece']),
     str(a3['ece_improvement_pct']) + "% better"],
    ["False Confidence Count",
     str(a3['baseline_false_confidence']),
     str(a3['guided_false_confidence']),
     str(a3['false_conf_reduction']) + " fewer"],
]

col_w = [28, 22, 24, 30]
sep   = "-+-".join("-" * w for w in col_w)
for i, row in enumerate(rows):
    line = " | ".join(str(cell).ljust(col_w[j]) for j, cell in enumerate(row))
    print("  " + line)
    if i == 0:
        print("  " + sep)

if a1.get("refiner_triggered", 0) > 0:
    rt = a1["refiner_triggered"]
    rc = a1.get("refiner_correct", 0)
    print(f"\n  Refiner: triggered {rt} times, resolved {rc} correctly ({rc/rt*100:.1f}%)")

# Save full report
full = {
    "dataset": "SVAMP", "seed": CONFIG["random_seed"],
    "pipeline": "LLM-SLM (70B verbal guide + 1B local solver)",
    "n_questions": n, "angle1": a1, "angle2": a2, "angle3": a3,
}
with open(CONFIG["report_file"], "w") as f:
    json.dump(full, f, indent=2)

print(f"\nAll results saved to {OUTPUT_DIR}")
print("Commit this notebook to preserve outputs.")


  SVAMP EVALUATION -- PAPER SUMMARY TABLE (LLM-SLM)
  Dataset  : SVAMP  |  Questions: 300  |  Seed: 42
  Guide    : Llama 70B via NVIDIA API (verbal plans, no arithmetic)
  Solver   : Llama 1.5B local

  Metric                       | Baseline               | Guided (LLM-SLM)         | Change                        
  -----------------------------+------------------------+--------------------------+-------------------------------
  Overall Accuracy             | 32.33%                 | 56.67%                   | +24.3 pts                     
  Local Compute Cost           | 5.0B param-passes      | 5.0B (+ 70B cloud guide) | Same local compute; gain from plan
  Wasted Votes                 | 780                    | 750                      | 30 fewer wasted passes        
  Vote Consistency             | 24.3%                  | 41.0%                    | 1.69x lift                    
  High-Agreement Questions     | 26                     | 70                       |              